# LA Studio voice-isolation - Spleeter 2-stem FP16

This notebook runs exactly `sherpa-onnx-spleeter-2stems-fp16` from the declared k2-fsa artifact on the temporary **Colab GPU worker**. It never uses API Gateway or a local LA Studio model.

The worker performs a CUDA startup probe before it prints a URL. It also sends long audio as bounded, overlapping segments, so the Spleeter FP16 CUDA convolution plan remains within the verified shape.

1. Choose **Runtime -> Change runtime type -> GPU**.
2. Run all cells. The final cell must print `startup probe: passed`.
3. Copy the printed URL and token to Dubbing -> Colab setup, then press **Check Colab**.


In [ ]:
!nvidia-smi
%pip install -q --upgrade --no-cache-dir "onnxruntime-gpu==1.21.0" "kaldi-native-fbank" "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3" "python-multipart==0.0.20"

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No Colab CUDA GPU is available. Select Runtime > Change runtime type > GPU, then restart and Run all.')
print('Colab CUDA:', torch.cuda.get_device_name(0))

!wget -q --show-progress -O /content/spleeter.tar.bz2 https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2
!tar -xjf /content/spleeter.tar.bz2 -C /content


In [ ]:
from hashlib import sha256
from pathlib import Path
from urllib.request import urlopen

MODEL_ID = "sherpa-onnx-spleeter-2stems-fp16"
WORKER_COMMIT = "eaabb045efa718c614d8155b5976fccb2e925a6b"  # audited exact worker revision
WORKERS = {
    "la_studio_separation_worker.py": (
        "notebooks/workers/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_WORKER.py",
        "307861926e13ff9849b04594074b573b1da063b1791c56dc2f502ab64991c5af"),
    "la_studio_separation_launcher.py": (
        "notebooks/workers/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_LAUNCHER.py",
        "d4c5d4756dd2d27ad2971c2ae338be1e92ae1ad790a6ef6d3ce07570bdfc8590"),
}
for destination, (relative_path, expected_sha256) in WORKERS.items():
    url = f"https://raw.githubusercontent.com/khoinguyen59/kova-video-studio/{WORKER_COMMIT}/{relative_path}"
    payload = urlopen(url, timeout=60).read()
    actual_sha256 = sha256(payload).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f"Worker integrity check failed for {relative_path}: {actual_sha256}")
    Path('/content', destination).write_bytes(payload)
print('Downloaded verified exact-model CUDA worker templates.')


In [ ]:
!python /content/la_studio_separation_launcher.py
